# 01 - Explore the Lodestar store

Sanity-check the enriched near-Earth store from Phase 0 / Phase 1 data layer: coverage, value, delta-v, and the value-vs-accessibility shortlist.

**Run the pipeline first** (`backend.ingest.sbdb`, `asterank`, `normalize`, then `backend.ranking.enrich`) so `data/processed/asteroids_enriched.parquet` exists. Every value and delta-v figure here is an *estimate with uncertainty*, never a measurement.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

store = Path('data/processed/asteroids_enriched.parquet')
if not store.exists():
    store = Path('../data/processed/asteroids_enriched.parquet')
df = pd.read_parquet(store)
print(f'{len(df):,} near-Earth asteroids x {len(df.columns)} columns')
df.head()

## Coverage and provenance

The honesty check: how much is measured vs assumed. Most objects have no measured spectrum and no measured size, so value leans on assumed type + H-derived diameter, all flagged. This is the gap Phase 4's ML narrows.

In [ ]:
n = len(df)
for label, mask in {
    'has a value estimate': df['value_usd'].notna(),
    '  value: measured spectral type': ~df['spec_is_assumed'].fillna(True),
    '  value: measured size': df['size_source'].eq('measured'),
    'delta-v from Benner table': df['dv_source'].eq('asterank-benner'),
    'delta-v from Hohmann proxy': df['dv_source'].eq('computed:hohmann-proxy'),
}.items():
    c = int(mask.sum())
    print(f'{label:34} {c:>7,}  ({c / n:.1%})')

In [ ]:
# compositional complex mix (mostly assumed S - honest about what we don't know)
ax = df['value_complex'].value_counts().plot(kind='bar', figsize=(6, 3),
                                             title='Assigned complex (C/S/M)')
ax.set_ylabel('count'); plt.tight_layout(); plt.show()
measured = df[~df['spec_is_assumed'].fillna(True)]
print(f'{len(measured):,} objects have a MEASURED spectral type')

## Value and accessibility distributions

Value spans many orders of magnitude (log it). Delta-v is the accessibility axis: lower is easier to reach.

In [ ]:
val = df.loc[df['value_usd'] > 0, 'value_usd']
dv = df.loc[df['dv_kms'].notna(), 'dv_kms']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(np.log10(val), bins=60)
axes[0].set_title('log10 estimated value (USD)'); axes[0].set_xlabel('log10 USD')
axes[1].hist(dv, bins=60)
axes[1].set_title('rendezvous delta-v (km/s)'); axes[1].set_xlabel('km/s')
plt.tight_layout(); plt.show()

In [ ]:
# the value x accessibility tradeoff the Phase 1 sliders will expose
sub = df[df['value_usd'].notna() & df['dv_kms'].notna()]
plt.figure(figsize=(7, 6))
plt.scatter(sub['dv_kms'], np.log10(sub['value_usd']), s=5, alpha=0.15)
plt.xlabel('delta-v (km/s)  -  lower = more accessible')
plt.ylabel('log10 estimated value (USD)')
plt.title('Value vs accessibility (each point an NEA estimate)')
plt.tight_layout(); plt.show()

In [ ]:
# the prospecting shortlist: accessible AND valuable. These should be real,
# recognizable mission targets (Apophis, Itokawa, 2008 EV5, ...).
cols = ['full_name', 'spec_type', 'value_complex', 'value_usd', 'value_low',
        'value_high', 'dv_kms', 'dv_source', 'spec_is_assumed']
accessible = sub[sub['dv_kms'] < sub['dv_kms'].quantile(0.10)]
accessible.sort_values('value_usd', ascending=False)[cols].head(15)